In [ ]:
from src.config import OllamaConfig

OLLAMA_HOST = "172.19.176.1"
OLLAMA_PORT = 11434
OLLAMA_URL = f"http://{OLLAMA_HOST}:{OLLAMA_PORT}/api/chat"

config = OllamaConfig(embedding_model="BAAI/bge-base-en", ollama_url=OLLAMA_URL)

print("Base directory:", config.base_dir)

config.ensure_dirs()

In [ ]:
from src.load_data import ensure_data_available

ensure_data_available(config=config)

print("Dataset ready")

In [ ]:
from src.load_data import load_embeddings

corpus, corpus_embeddings = load_embeddings(config=config)

In [ ]:
from src.retriever import Retriever
from src.reranker import Reranker
from src.generator import OllamaGenerator

retriever = Retriever(config=config)
reranker = Reranker(config=config)
generator = OllamaGenerator(config=config)

In [ ]:
query = "What is the capital of France?"
candidates = 100
top_k = 3

retrieved_passages, _, candidates_idx = retriever.retrieve(
    query=query, 
    corpus=corpus,
    faiss_index=corpus_embeddings, 
    top_k=candidates)

context, _ = reranker.rerank_and_get_top_k(
    query=query, 
    corpus=corpus, 
    candidates_idx=candidates_idx,
    top_k=top_k)

answer = generator.generate(query=query, passages=context)

print("\nGenerated Answer:\n", answer)

In [ ]:
print("\n🔍 Used Context Passages:\n")
for i,p in enumerate(context, 1):
    print(f"{i}. {p[:200].replace(chr(10),' ')}...\n")